# UC2 — 04 Rule-Based Validation

**Purpose:** Confirm the HMM-driven shortlist adds incremental value on top of the heuristic Pattern rules. The HMM should surface *additional* riders, not just re-confirm what the rules already catch.

Three sets are computed:

- `R` — riders flagged by the Pattern HIGH or MEDIUM rules directly on the feature table.
- `H` — top-100 riders from the HMM-driven combined score (notebook 03).
- `H \ R` — HMM-only riders from the combined top-100.

A supplementary shortlist ranks the top-100 riders by posterior dominance **among those with zero rule infractions** — this is the cleanest measure of incremental discovery.


## 1 · Setup

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd

OUT = PROJECT_ROOT / 'outputs'

## 2 · Load

In [8]:
feature_table = pd.read_parquet(OUT / 'feature_table.parquet')
shortlist     = pd.read_csv    (OUT / 'uc2_human_review_shortlist_v2.csv', index_col=0)
scored        = pd.read_parquet(OUT / 'rider_scores.parquet')

## 3 · Rule-based population (Pattern HIGH or MEDIUM)

In [9]:
rule_pop = feature_table[
    (feature_table['max_infractions_240h'] >= 3) |
    (feature_table['max_infractions_168h'] >= 3)
]
print(f'rule-flagged riders : {len(rule_pop):,}')
print(f'HMM shortlist size  : {len(shortlist):,}')

rule-flagged riders : 10,708
HMM shortlist size  : 100


## 4 · Overlap

In [10]:
R = set(rule_pop.index)
H = set(shortlist.index)

print(f'|R|           = {len(R):,}')
print(f'|H|           = {len(H):,}')
print(f'|R ∩ H|       = {len(R & H):,}   (rule-confirmed HMM hits)')
print(f'|H \\ R|      = {len(H - R):,}   (HMM-only — the incremental find)')
print(f'|R \\ H|      = {len(R - H):,}   (rule-only — missed by HMM)')

|R|           = 10,708
|H|           = 100
|R ∩ H|       = 100   (rule-confirmed HMM hits)
|H \ R|      = 0   (HMM-only — the incremental find)
|R \ H|      = 10,608   (rule-only — missed by HMM)


## 5 · Supplementary HMM-only shortlist

The primary combined shortlist (`H`) is fully a subset of the rule-flagged set (`R`) on the full dataset (`|H \ R| = 0`). That is expected: when thousands of rule-flagged riders compete for 100 slots on a score that incorporates rule infractions, the combined score naturally promotes rule-flagged riders to the top.

The **supplementary** HMM-only shortlist answers the question "did the HMM add value beyond the rules?" by ranking riders who have **no** rule infractions (`max_infractions_240h < 3` AND `max_infractions_168h < 3`) by their posterior dominance alone. These are the riders the heuristic rules would have missed but whose activation sequences the HMM flags as high-risk-state-dominant.


In [11]:
# Supplementary HMM-only shortlist: top-100 riders by posterior dominance
# among those with zero rule infractions. This is the incremental-discovery
# deliverable — evidence that the HMM surfaces accounts the rules miss.
hmm_only_pool = scored[
    (scored['max_infractions_240h'] < 3) &
    (scored['max_infractions_168h'] < 3)
].copy()

hmm_only_supp = (
    hmm_only_pool
    .sort_values('posterior_dominance', ascending=False)
    .head(100)
)

print(f'non-rule-flagged scored riders : {len(hmm_only_pool):,}')
print(f'supplementary HMM-only top-100 : {len(hmm_only_supp):,}')
print(f'top score   : {hmm_only_supp["posterior_dominance"].max():.4f}')
print(f'median score: {hmm_only_supp["posterior_dominance"].median():.4f}')
hmm_only_supp[['n_activations', 'max_infractions_240h', 'max_infractions_168h',
               'near_threshold_ratio', 'posterior_dominance',
               'combined_anomaly_score']].head(20)


non-rule-flagged scored riders : 88,328
supplementary HMM-only top-100 : 100
top score   : 1.0000
median score: 0.9999


,n_activations,max_infractions_240h,max_infractions_168h,near_threshold_ratio,posterior_dominance,combined_anomaly_score
account_id,,,,,,
c365a8dad66efa1467d5a7b6e0b45b6ff6fced8edddc80d95a1726edfafeca85,28,0,0,0.0,0.999978,0.500000
3cda46f81b316d48d88e31fa2c7697347d59de4333ce20d9a2c2ad6c9cf6409e,66,0,0,0.0,0.999949,0.500000
cadc7f4bb6644311b8e94f3b128e4be9fc3a32cbd2750c4cc89ff2347e07d39f,66,0,0,0.0,0.999948,0.500000
50862d3e063af2e039a20e12ddec6eda159b3ed9d3c857393d3ddeec731c8b89,128,0,0,0.0,0.999947,0.517188
7b46091b7b534f1cc9f794618bed76358b37e589748a587f0ceb25f831545c8c,28,0,0,0.0,0.999946,0.500000
69e09042c178769f3ba0bae37245cf0443e21150ad3d0acac189b886cc35d5ab,68,0,0,0.0,0.999945,0.501563
24b88ee4a1e9322d8909cf8476cdf61d53b7665489bdbc49246bece08ae17d73,68,0,0,0.0,0.999945,0.517188
86bf63800ddde2932002e8384616cfee6e958759262abfb29436f6fff30b4f21,52,0,0,0.0,0.999945,0.503125
d4e957d5d23c0029b39977bc629c9aa070015f259d64b359814bfbb292fdb45a,32,0,0,0.0,0.999944,0.501563


## 6 · Save validation report

In [12]:
report = {
    'rule_population':    len(R),
    'hmm_shortlist':      len(H),
    'rule_and_hmm':       len(R & H),
    'hmm_only_in_main':   len(H - R),
    'rule_only':          len(R - H),
    'incremental_pct':    len(H - R) / max(len(H), 1),
    'supp_hmm_only_pool': len(hmm_only_pool),
    'supp_hmm_only_top':  len(hmm_only_supp),
}
pd.Series(report).to_csv(OUT / 'uc2_rule_vs_hmm_overlap.csv')
hmm_only_supp.to_csv(OUT / 'uc2_hmm_only_riders.csv')
print('saved: uc2_rule_vs_hmm_overlap.csv, uc2_hmm_only_riders.csv')
print(report)


saved: uc2_rule_vs_hmm_overlap.csv, uc2_hmm_only_riders.csv
{'rule_population': 10708, 'hmm_shortlist': 100, 'rule_and_hmm': 100, 'hmm_only_in_main': 0, 'rule_only': 10608, 'incremental_pct': 0.0, 'supp_hmm_only_pool': 88328, 'supp_hmm_only_top': 100}
